# Lab 5: Deep Learning & LLMs for NLP

**Course:** Natural Language Processing


**Objectives:**
- Understand RNN, LSTM, GRU architectures for sequence modeling
- Use pre-trained Transformers for NER
- Interact with LLMs via API for text generation

---

## Instructions

1. Complete all exercises marked with `# YOUR CODE HERE`
2. **Answer all written questions** in the designated markdown cells
3. Save your completed notebook
4. **Push to your Git repository and send the link to: yoroba93@gmail.com**

---

## Lab Structure

| Part | Model | Task |
|------|-------|------|
| A | RNN | Character-level Language Model |
| B | LSTM | Sentiment Analysis |
| C | GRU | News Classification |
| D | Transformer | Named Entity Recognition |
| E | LLM (Mistral) | Text Generation & QA |

---

## Setup

In [2]:
# Install required libraries (uncomment if needed)
!pip install torch transformers datasets==3.6.0 requests numpy pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.11.0+cu128


---

# PART A: RNN - Character-Level Language Model (10 min)

**Use Case:** Predict the next character for autocomplete.

**Dataset:** Tiny Shakespeare

In [4]:
# Load Tiny Shakespeare dataset
from datasets import load_dataset

shakespeare = load_dataset("karpathy/tiny_shakespeare", split="train")
text = shakespeare['text'][0][:10000]  # Use first 10K chars for speed

print(f"Text length: {len(text)} characters")
print(f"Sample: {text[:200]}")

README.md:   0%|          | 0.00/6.10k [00:00<?, ?B/s]

tiny_shakespeare.py:   0%|          | 0.00/3.73k [00:00<?, ?B/s]

The repository for karpathy/tiny_shakespeare contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/karpathy/tiny_shakespeare.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

Text length: 10000 characters
Sample: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [5]:
# Create character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars[:30])}...")

Vocabulary size: 57
Characters: 
 !',-.:;?ABCDEFHIJLMNOPRSTUVW...


In [6]:
# Prepare sequences
seq_length = 30
X, y = [], []

for i in range(len(text) - seq_length):
    X.append([char_to_idx[c] for c in text[i:i+seq_length]])
    y.append(char_to_idx[text[i+seq_length]])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(f"Sequences: {X.shape[0]}, Sequence length: {seq_length}")

Sequences: 9970, Sequence length: 30


In [7]:
# Simple RNN model
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Last timestep
        return out

# Create model
rnn_model = CharRNN(vocab_size, embed_dim=32, hidden_dim=64).to(device)
print(f"RNN Parameters: {sum(p.numel() for p in rnn_model.parameters()):,}")

RNN Parameters: 11,801


### Exercise A.1: Train the RNN

In [8]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# TODO: Complete the training loop

# Hyperparameters
batch_size = 128
epochs = 5
learning_rate = 0.005  # YOUR CHOICE: 0.001-0.01

# DataLoader
dataset = TensorDataset(X[:5000], y[:5000])  # Use subset for speed
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=learning_rate)

# Training loop
losses = []
for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # YOUR CODE HERE
        # 1. Zero gradients
        optimizer.zero_grad()
        # 2. Forward pass
        output = rnn_model(batch_X)
        # 3. Compute loss
        loss = criterion(output, batch_y)
        # 4. Backward pass
        loss.backward()
        # 5. Update weights
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 3.0878
Epoch 2/5, Loss: 2.5365
Epoch 3/5, Loss: 2.3219
Epoch 4/5, Loss: 2.1947
Epoch 5/5, Loss: 2.1099


In [9]:
# Generate text
def generate_text(model, start_str, length=100):
    model.eval()
    chars_generated = list(start_str)
    input_seq = [char_to_idx.get(c, 0) for c in start_str[-seq_length:]]

    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([input_seq[-seq_length:]]).to(device)
            output = model(x)
            pred_idx = torch.argmax(output, dim=1).item()
            chars_generated.append(idx_to_char[pred_idx])
            input_seq.append(pred_idx)

    return ''.join(chars_generated)

# Test generation
print("Generated text:")
print(generate_text(rnn_model, "To be or not", length=100))

Generated text:
To be or not the the the the the the the the the the the the the the the the the the the the the the the the the


---

# PART B: LSTM - Sentiment Analysis

**Use Case:** Classify movie review sentiment.

**Dataset:** IMDB Reviews

In [10]:
# Load IMDB dataset
imdb = load_dataset("stanfordnlp/imdb")

# Small sample for quick training
train_texts = imdb['train']['text'][:1000]
train_labels = imdb['train']['label'][:1000]
test_texts = imdb['test']['text'][:200]
test_labels = imdb['test']['label'][:200]

print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train: 1000, Test: 200


In [11]:
# Simple tokenization and vocabulary
from collections import Counter
import re

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())[:100]  # Max 100 tokens

# Build vocabulary from training data
all_tokens = [tok for text in train_texts for tok in tokenize(text)]
vocab = {word: idx+2 for idx, (word, _) in enumerate(Counter(all_tokens).most_common(5000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 5002


In [12]:
# Encode texts
def encode_text(text, max_len=100):
    tokens = tokenize(text)
    encoded = [vocab.get(t, 1) for t in tokens]  # 1 = UNK
    padded = encoded[:max_len] + [0] * (max_len - len(encoded))
    return padded[:max_len]

X_train = torch.tensor([encode_text(t) for t in train_texts])
y_train = torch.tensor(train_labels)
X_test = torch.tensor([encode_text(t) for t in test_texts])
y_test = torch.tensor(test_labels)

print(f"Train shape: {X_train.shape}")

Train shape: torch.Size([1000, 100])


### Exercise B.1: Complete the LSTM Model

In [13]:
# TODO: Complete the LSTM classifier

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # YOUR CODE HERE: Define LSTM layer
        # Hint: nn.LSTM(input_size, hidden_size, batch_first=True)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)

        # YOUR CODE HERE: Pass through LSTM and get final hidden state
        # Hint: lstm_out, (hidden, cell) = self.lstm(x)
        lstm_out, (hidden, cell) = self.lstm(x)

        out = self.fc(hidden.squeeze(0))  # Use last hidden state
        return out

# Create model
lstm_model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden_dim=64,
    num_classes=2
).to(device)

print(f"LSTM Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

LSTM Parameters: 353,538


In [14]:
# Quick training
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

# Train for 3 epochs
for epoch in range(3):
    lstm_model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = lstm_model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Evaluate
lstm_model.eval()
with torch.no_grad():
    test_output = lstm_model(X_test.to(device))
    preds = torch.argmax(test_output, dim=1).cpu()
    acc = (preds == y_test).float().mean()
    print(f"\nTest Accuracy: {acc:.4f}")

Epoch 1, Loss: 0.4390
Epoch 2, Loss: 0.0047
Epoch 3, Loss: 0.0004

Test Accuracy: 1.0000


---

# PART C: GRU - News Classification

**Use Case:** Classify news articles by topic.

**Why GRU?** Fewer parameters than LSTM, faster training.

In [15]:
# Load AG News
ag_news = load_dataset("sh0416/ag_news")
ag_train = ag_news['train'].shuffle(seed=42).select(range(2000))
ag_test = ag_news['test'].shuffle(seed=42).select(range(500))

ag_labels = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
print(f"Classes: {list(ag_labels.values())}")

README.md:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 33.7MB            

train.jsonl: downloading bytes:           |  0.00B            

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Classes: ['World', 'Sports', 'Business', 'Sci/Tech']


### Exercise C.1: Build GRU Classifier

In [16]:
# TODO: Create a GRU classifier (similar to LSTM but using nn.GRU)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # YOUR CODE HERE: Define GRU layer
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        # YOUR CODE HERE: GRU forward pass
        # Note: GRU returns (output, hidden) - no cell state unlike LSTM
        _, hidden = self.gru(x)

        out = self.fc(hidden.squeeze(0))  # Use last hidden state
        return out

# Build vocabulary and encode (reuse tokenize function)

# Extract text and labels from the Dataset objects
ag_train_texts = ag_train['description']
ag_train_labels = ag_train['label']
ag_test_texts = ag_test['description']
ag_test_labels = ag_test['label']

ag_tokens = [tok for text_item in ag_train_texts for tok in tokenize(text_item)]
ag_vocab = {word: idx+2 for idx, (word, _) in enumerate(Counter(ag_tokens).most_common(5000))}
ag_vocab['<PAD>'] = 0
ag_vocab['<UNK>'] = 1

def encode_ag(text, vocab, max_len=50):
    tokens = tokenize(text)
    encoded = [vocab.get(t, 1) for t in tokens]
    return (encoded[:max_len] + [0] * max_len)[:max_len]

X_ag_train = torch.tensor([encode_ag(text_item, ag_vocab) for text_item in ag_train_texts])
y_ag_train = torch.tensor(ag_train_labels)
X_ag_test = torch.tensor([encode_ag(text_item, ag_vocab) for text_item in ag_test_texts])
y_ag_test = torch.tensor(ag_test_labels)

print(f"AG News - Train: {X_ag_train.shape}, Test: {X_ag_test.shape}")

AG News - Train: torch.Size([2000, 50]), Test: torch.Size([500, 50])


In [17]:
# Create and train GRU model
gru_model = GRUClassifier(
    vocab_size=len(ag_vocab),
    embed_dim=64,
    hidden_dim=64,
    num_classes=4
).to(device)

print(f"GRU Parameters: {sum(p.numel() for p in gru_model.parameters()):,}")
print(f"(Compare to LSTM: GRU has fewer parameters!)")

GRU Parameters: 345,348
(Compare to LSTM: GRU has fewer parameters!)


---

# PART D: Transformer - Named Entity Recognition

**Use Case:** Extract entities from text.

**Dataset:** CoNLL-2003

In [18]:
# Use pre-trained NER model from Hugging Face
from transformers import pipeline

# Load NER pipeline (uses BERT-based model)
print("Loading NER model...")
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
print("Model loaded!")

Loading NER model...


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded!


In [19]:
# Example NER
text = "Apple Inc. was founded by Steve Jobs in Cupertino, California. Tim Cook is the current CEO."

entities = ner_pipeline(text)
print(f"Text: {text}\n")
print("Entities found:")
for ent in entities:
    print(f"  {ent['word']:20} -> {ent['entity_group']:10} (score: {ent['score']:.3f})")

Text: Apple Inc. was founded by Steve Jobs in Cupertino, California. Tim Cook is the current CEO.

Entities found:
  Apple Inc            -> ORG        (score: 0.999)
  Steve Jobs           -> PER        (score: 0.903)
  Cupertino            -> LOC        (score: 0.998)
  California           -> LOC        (score: 0.999)
  Tim Cook             -> PER        (score: 1.000)


### Exercise D.1: NER on Your Own Texts

In [20]:
# TODO: Write 3 sentences and extract entities
# Include: people, organizations, locations

my_sentences = [
    "Elon Musk visited the SpaceX headquarters in Texas last week.",  # YOUR SENTENCE 1
    "Google announced a new AI initiative in Mountain View, California.",  # YOUR SENTENCE 2
    "Dr. Sarah Connor, a lead researcher at NASA, presented her findings in Washington D.C.",  # YOUR SENTENCE 3
]

for sent in my_sentences:
    if sent != "___":
        print(f"\nText: {sent}")
        entities = ner_pipeline(sent)
        for ent in entities:
            print(f"  {ent['word']:20} -> {ent['entity_group']}")


Text: Elon Musk visited the SpaceX headquarters in Texas last week.
  Elon Musk            -> ORG
  SpaceX               -> ORG
  Texas                -> LOC

Text: Google announced a new AI initiative in Mountain View, California.
  Google               -> ORG
  AI                   -> MISC
  Mountain View        -> LOC
  California           -> LOC

Text: Dr. Sarah Connor, a lead researcher at NASA, presented her findings in Washington D.C.
  Sarah Connor         -> PER
  NASA                 -> ORG
  Washington D. C      -> LOC


---

# PART E: LLM - Text Generation with Mistral API

**Use Case:** Conversational AI and Question Answering.

**Setup:** Get a free API key from https://console.mistral.ai/

In [21]:
# TODO: Enter your Mistral API key
# Get free key at: https://console.mistral.ai/

MISTRAL_API_KEY = "ZxSmP4JI0kWxhTGFddF526FTpmAlA9RV"  # YOUR API KEY HERE

In [22]:
import requests

def query_mistral(prompt, max_tokens=150):
    """Query Mistral API."""
    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "mistral-small-latest",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens
    }

    response = requests.post(url, headers=headers, json=data)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        return f"Error: {response.status_code} - {response.text}"

# Test (only if API key is set)
if MISTRAL_API_KEY != "___":
    response = query_mistral("What is NLP in one sentence?")
    print(f"Mistral: {response}")
else:
    print("Please set your MISTRAL_API_KEY above.")

Mistral: Natural Language Processing (NLP) is a field of AI that enables computers to understand, interpret, and generate human language.


### Exercise E.1: Compare LLM with Traditional Models

In [23]:
# TODO: Ask Mistral to perform sentiment analysis and compare with our LSTM

test_review = "This movie was absolutely terrible. The acting was bad and the plot made no sense."

# LLM approach
if MISTRAL_API_KEY != "ZxSmP4JI0kWxhTGFddF526FTpmAlA9RV":
    prompt = f"""Classify the sentiment of this review as 'positive' or 'negative'.
Just respond with one word.

Review: {test_review}

Sentiment:"""

    llm_result = query_mistral(prompt, max_tokens=10)
    print(f"LLM Sentiment: {llm_result}")

# Traditional LSTM approach (if model trained)
try:
    encoded = torch.tensor([encode_text(test_review)]).to(device)
    lstm_model.eval()
    with torch.no_grad():
        lstm_pred = torch.argmax(lstm_model(encoded)).item()
    print(f"LSTM Sentiment: {'positive' if lstm_pred == 1 else 'negative'}")
except:
    print("LSTM model not available")

LSTM Sentiment: negative


In [24]:
# TODO: Use LLM for summarization (something traditional models can't easily do)

long_text = """
Natural language processing (NLP) is a subfield of linguistics, computer science,
and artificial intelligence concerned with the interactions between computers and
human language, in particular how to program computers to process and analyze large
amounts of natural language data. The result is a computer capable of understanding
the contents of documents, including the contextual nuances of the language within them.
"""

if MISTRAL_API_KEY != "ZxSmP4JI0kWxhTGFddF526FTpmAlA9RV":
    summary_prompt = f"Summarize this in one sentence:\n\n{long_text}"
    summary = query_mistral(summary_prompt, max_tokens=50)
    print(f"Summary: {summary}")

---

## Final Written Questions (Personal Interpretation)

Answer these questions based on YOUR experiments:

### Question 1: Model Architecture Comparison

Compare the parameter counts you observed:
- RNN: 11,801 parameters
- LSTM: 353,538 parameters  
- GRU: 345,348 parameters

**Why does LSTM have more parameters than GRU?** (Hint: think about gates)

**YOUR ANSWER:**
LSTM (Long Short-Term Memory) models have more parameters than GRU (Gated Recurrent Unit) models primarily because LSTMs have a more complex gating mechanism. LSTMs typically use three gates: an input gate, a forget gate, and an output gate, along with a cell state. Each of these gates requires its own weight matrices and bias vectors for the input and the previous hidden state.

GRU, on the other hand, combines the input and forget gates into a single update gate and also has a reset gate. It doesn't have a separate cell state like LSTM. This simpler architecture means GRUs require fewer parameters, making them computationally less expensive and faster to train, while often achieving comparable performance to LSTMs on many tasks.

**YOUR ANSWER:**

LSTM would perform better than a vanilla RNN for sentiment analysis on long reviews due to its ability to address the **vanishing gradient problem**.

The **vanishing gradient problem** occurs in vanilla RNNs when gradients become extremely small during backpropagation through many time steps. This means that the influence of earlier inputs on the current output diminishes over long sequences, making it difficult for the network to learn long-term dependencies. For a long movie review, important context from the beginning of the review might be lost by the time the RNN processes the end.

LSTMs mitigate this problem using **gates** (input, forget, and output gates) and a **cell state**. The cell state acts as a memory unit that can retain information over extended periods. The gates control the flow of information into and out of this cell state:
*   **Forget gate:** Decides what information to discard from the cell state.
*   **Input gate:** Decides what new information to store in the cell state.
*   **Output gate:** Decides what part of the cell state to output as the hidden state.

This controlled flow allows LSTMs to selectively remember or forget information, enabling them to capture long-term dependencies crucial for understanding the overall sentiment of a long review, where important context might be spread across the text.

### Question 2: RNN vs LSTM for Long Sequences

**Why would LSTM perform better than vanilla RNN for sentiment analysis on long reviews?** Explain the vanishing gradient problem.

**YOUR ANSWER:**

1.  **What can LLMs do that LSTM/GRU cannot?**
    LLMs (Large Language Models) can perform a much wider array of complex natural language tasks with remarkable fluency and generalizability that traditional LSTM/GRU models struggle with or cannot do at all. These include:
    *   **Complex Text Generation:** Generating coherent, contextually relevant, and creative text (e.g., poems, stories, detailed explanations, code).
    *   **Advanced Summarization:** Producing high-quality, abstractive summaries of long documents, capturing main ideas rather than just extracting sentences.
    *   **Question Answering (Open-Domain):** Answering questions based on general knowledge or provided context without explicit training for specific question types.
    *   **Zero-Shot/Few-Shot Learning:** Performing tasks they haven't been explicitly trained on, simply by understanding the instructions in the prompt.
    *   **Conversational AI:** Engaging in dynamic, multi-turn conversations.
    *   **Translation & Multilingual Tasks:** Handling multiple languages effectively.
    *   **Reasoning and Inference:** Performing some level of logical reasoning and inference based on textual input.

2.  **What are the disadvantages of using LLM APIs?**
    *   **Cost:** API usage typically incurs costs per token, which can quickly become expensive for high-volume or complex tasks.
    *   **Latency:** API calls involve network requests, which introduce latency compared to running local models. This can be a concern for real-time applications.
    *   **Privacy/Data Security:** Sending sensitive data to third-party APIs can raise privacy and data security concerns, as the data is processed on external servers.
    *   **Lack of Control/Transparency:** Users have less control over the model's internal workings, biases, and how it generates responses. Outputs can be unpredictable or unexplainable.
    *   **Dependence on External Services:** Reliance on an external provider means potential service outages, changes in API terms, or deprecation.
    *   **Potential for Hallucinations:** LLMs can sometimes generate factually incorrect or nonsensical information.

3.  **When would you choose a traditional model over an LLM?**
    You would choose a traditional model (like RNN, LSTM, GRU, or even simpler machine learning models) over an LLM in several scenarios:
    *   **Resource Constraints:** When computational resources (GPU, memory) are limited, traditional models are much lighter and faster to train and deploy.
    *   **Specific, Well-Defined Tasks:** For highly specific tasks like basic sentiment classification, spam detection, or named entity recognition on a known dataset, a well-tuned traditional model can achieve high accuracy with less overhead.
    *   **Interpretability Requirements:** If model interpretability is crucial (e.g., in medical or financial domains), traditional models are often easier to understand and debug.
    *   **Data Privacy/Security:** When sensitive data cannot leave an on-premise environment or specific security protocols must be strictly adhered to.
    *   **Cost Efficiency:** For tasks that can be solved effectively by simpler models, using an LLM API would be overkill and unnecessarily expensive.
    *   **Dataset Size:** If you have a relatively small, task-specific dataset, fine-tuning a traditional model might be more effective and less prone to overfitting than trying to adapt an LLM.

### Question 3: Traditional Models vs LLMs

Based on your experiments:
1. **What can LLMs do that LSTM/GRU cannot?**
2. **What are the disadvantages of using LLM APIs?** (Think: cost, latency, privacy)
3. **When would you choose a traditional model over an LLM?**

**YOUR ANSWER:**

1. LLM advantages: ...

2. LLM disadvantages: ...

3. When to use traditional models: ...

---

## Summary

| Model | Strength | Weakness | Best For |
|-------|----------|----------|----------|
| RNN | Simple, fast | Vanishing gradients | Short sequences |
| LSTM | Long-term memory | More parameters | Long text classification |
| GRU | Efficient, fast | Less expressive | When speed matters |
| Transformer | Parallel, contextual | Expensive | NER, QA, many tasks |
| LLM | Versatile, zero-shot | API cost, latency | Complex reasoning |

---

## Submission

- [ ] All code exercises completed
- [ ] All written questions answered
- [ ] Mistral API tested (or explained why not)
- [ ] **Push to Git and send link to: yoroba93@gmail.com**